# Solution of linear systems and interpolations

Using numerical-computational techniques to solve linear systems and interpolations 💻

## All imports:

Import the dependencies required for the project

In [119]:
import numpy as np
import sys
from typing import Tuple, Callable

## Solving Linear Systems:

Solving linear systems using direct and iterative numerical methods

### Overall parameters

Pre-defined parameters for analysis

In [120]:
def create_dominant_diagonal_matrix(n: int) -> np.ndarray:
    matrix = np.random.randint(low=0, high=10, size=(n, n))
    for i in range(n):
        matrix[i,i] = np.sum(abs(matrix[i, :])) + np.sum(abs(matrix[:, i])) + 1
    return matrix

A : np.ndarray = np.array([
        [10, 2, 1],
        [1, 5, 1],
        [2, 3, 10]
    ])

F : np.ndarray= np.array([
        [32,  5,  7,  0],
        [ 3, 30,  6,  1],
        [ 7,  5, 54,  8],
        [ 1,  9,  8, 30]
    ])

C : np.ndarray= np.array([
        [45,  2,  5,  6,  0],
        [ 9, 37,  1,  7,  3],
        [ 6,  1, 49,  8,  5],
        [ 8,  9,  7, 55,  1],
        [ 0,  0,  3,  2, 31]
    ])

B3 : np.ndarray = np.array([7, -8, 6])
B4 : np.ndarray = np.array([1,2,3,4])
B5 : np.ndarray = np.array([1,2,3,4,5])

epsilon = 10**(-16)

### Auxiliary Functions

Auxiliary functions are utilities that perform specific and repetitive tasks, simplifying the main code. Below are the functions used in this project:

In [121]:
def dominant_line(matrix: np.ndarray) -> bool:
    for i in range(matrix.shape[0]):
        diagonal = np.abs(matrix[i, i])
        sum_line = np.sum(np.abs(matrix[i, :])) - diagonal
        if diagonal < sum_line:
            return False
    return True

def dominant_column(matrix: np.ndarray) -> bool:
    for i in range(matrix.shape[1]):
        diagonal = np.abs(matrix[i, i])
        sum_column = np.sum(np.abs(matrix[:, i])) - diagonal
        if diagonal < sum_column:
            return False
    return True

def dominant_diagonal(matrix: np.ndarray) -> bool:
    return dominant_line(matrix) or dominant_column(matrix)

def absolute_distance(new: np.ndarray, old: np.ndarray) -> np.float64:
    return np.max(abs(new-old))

def relative_distance(new: np.ndarray, old: np.ndarray, epsilon: float = 0) -> np.float64:
    ad = absolute_distance(new,old)
    new_max = np.max(abs(new))
    
    if new_max <= epsilon:
        new_max+=sys.float_info.epsilon
    
    return ad/new_max

def residue(A: np.ndarray, b: np.ndarray, x: np.ndarray) -> np.float64:
    return np.max(abs(b - (A @ x)))


### Gauss-Elimination

In [122]:
def gauss_scaling(A: np.ndarray, b: np.ndarray) -> Tuple[np.ndarray, np.ndarray, float]:
    """Performs Gaussian elimination with partial pivoting and scaling on a linear system.

    Transforms the coefficient matrix into upper triangular form using row operations,
    while maintaining numerical stability through partial pivoting. Also computes the
    determinant of the matrix as a side product.

    Args:
        A: Square coefficient matrix of the linear system (n x n numpy array).
        b: Right-hand side vector of the linear system (n-dimensional numpy array).

    Returns:
        Tuple containing:
        - A_new: Upper triangular matrix after elimination
        - b_new: Modified right-hand side vector after elimination
        - det: Determinant of the original matrix (product of diagonal elements with sign changes)

    Raises:
        ValueError: If the matrix is singular (no unique solution exists).

    Note:
        Modifies the input matrix and vector during the elimination process.
        The determinant is computed as a side product of the elimination steps.
    """
    n = len(b)
    det = 1.0

    for k in range(n):
        pivot_selection = np.argmax(abs(A[k:, k])) + k

        if A[pivot_selection, k] < sys.float_info.epsilon:
            raise ValueError("Matrix is singular (no unique solution).")

        if pivot_selection != k:
            A[[k, pivot_selection]] = A[[pivot_selection, k]]
            b[[k, pivot_selection]] = b[[pivot_selection, k]]
            det *= -1

        for i in range(k + 1, n):
            factor = A[i, k] / A[k, k]
            A[i, k:] -= factor * A[k, k:]
            b[i] -= factor * b[k]

    det *= np.prod(np.diagonal(A))
    return A, b, det

def retrosubstitution(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Solves an upper triangular linear system using backward substitution.

    Args:
        A: Upper triangular coefficient matrix (n x n numpy array).
        b: Right-hand side vector (n-dimensional numpy array).

    Returns:
        np.ndarray: Solution vector x that satisfies Ax = b.

    Raises:
        ValueError: If any diagonal element is zero (matrix is singular).
    """
    n = len(b)
    x = np.zeros(n)
    
    for i in reversed(range(n)):
        soma = np.dot(A[i, i+1:], x[i+1:])
        x[i] = (b[i] - soma) / A[i, i]
    
    return x

def gauss_elimination_method(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Solves a system of linear equations using Gaussian elimination with partial pivoting.

    Performs the complete solution process:
    1. Transforms the matrix to upper triangular form with partial pivoting
    2. Checks for singularity (zero determinant)
    3. Solves the triangular system using backward substitution

    Args:
        A: Square coefficient matrix of the linear system (n x n numpy array).
        b: Right-hand side vector of the linear system (n-dimensional numpy array).

    Returns:
        np.ndarray: Solution vector x that satisfies Ax = b.

    Raises:
        ValueError: If the matrix is singular (determinant is zero) or if any diagonal
                   element becomes zero during elimination.
    """

    A = A.astype(float)
    b = b.astype(float)
    
    A_new, b_new, det = gauss_scaling(A,b)

    if det == 0:
        raise ValueError("No single solution")
    
    return retrosubstitution(A_new, b_new)


### LU Factorization Method

In [123]:
def lu_decomposition(A):
    """Performs LU decomposition with partial pivoting on a square matrix.

    Decomposes a matrix A into PA = LU, where:
    - P is a permutation matrix
    - L is a lower triangular matrix with unit diagonal
    - U is an upper triangular matrix

    Args:
        A: Square coefficient matrix to decompose (n x n numpy array).

    Returns:
        Tuple containing three numpy arrays:
        - P: Permutation matrix representing row exchanges
        - L: Lower triangular matrix with ones on diagonal
        - U: Upper triangular matrix

    Raises:
        ValueError: If the matrix is singular (no unique decomposition exists).

    Note:
        Uses partial pivoting for numerical stability.
        The decomposition satisfies PA = LU.
    """
    A = A.astype(float)
    n = A.shape[0]
    
    L = np.eye(n)
    U = A.copy()
    P = np.eye(n)

    for k in range(n):
       
        pivot_row = np.argmax(np.abs(U[k:, k])) + k
        
        if U[pivot_row, k] == 0:
            raise ValueError("Matrix is singular (no unique solution).")
        
       
        if pivot_row != k:
            U[[k, pivot_row], :] = U[[pivot_row, k], :]
            P[[k, pivot_row], :] = P[[pivot_row, k], :]
            if k > 0:
                L[[k, pivot_row], :k] = L[[pivot_row, k], :k]
    
        for i in range(k + 1, n):
            fator = U[i, k] / U[k, k]
            L[i, k] = fator
            U[i, :] -= fator * U[k, :]

    return P, L, U

def lu_method(A, b):
    """Solves a linear system using LU decomposition with partial pivoting.

    Solves Ax = b by:
    1. Decomposing A into PA = LU
    2. Solving Ly = Pb (forward substitution)
    3. Solving Ux = y (backward substitution)

    Args:
        A: Square coefficient matrix of the linear system (n x n numpy array).
        b: Right-hand side vector of the linear system (n-dimensional numpy array).

    Returns:
        np.ndarray: Solution vector x that satisfies Ax = b.

    Raises:
        ValueError: If the matrix is singular (no unique solution exists).

    Note:
        More efficient than Gaussian elimination when solving multiple systems
        with the same coefficient matrix but different right-hand sides.
    """
    P, L, U = lu_decomposition(A)
    Pb = P @ b
    
    n = A.shape[0]
    
    
    y = np.zeros_like(b, dtype=float)
    for i in range(n):
        y[i] = Pb[i] - L[i, :i] @ y[:i]
        
    
    x = np.zeros_like(b, dtype=float)
    for i in reversed(range(n)):
        x[i] = (y[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
        
    return x


### Gauss-Jacobi

In [124]:
def gauss_jacobi_method(A: np.ndarray, b: np.ndarray, max_iterations: int = 100, epsilon: float = sys.float_info.epsilon) -> np.ndarray:
    """Solves a system of linear equations using the Gauss-Jacobi iterative method.

    The Gauss-Jacobi method is an iterative algorithm for solving systems of linear equations
    where the matrix is strictly diagonally dominant or symmetric and positive definite.
    At each iteration, it uses the solution from the previous iteration to compute new values.

    Args:
        A: A square numpy array representing the coefficient matrix of the linear system.
            Must be strictly diagonally dominant for convergence guarantee.
            Shape: (n, n) where n is the number of equations.
        b: A numpy array representing the right-hand side vector of the linear system.
            Shape: (n,) where n matches the matrix dimension.
        max_iterations: Maximum number of iterations to perform before giving up.
            Default: 100.
        epsilon: The tolerance for determining convergence. The algorithm stops when either
            the absolute or relative distance between iterations is less than epsilon.
            Default: system's float epsilon.

    Returns:
        A numpy array representing the solution vector x that satisfies Ax = b.
        Shape: (n,) matching the input dimensions.

    Raises:
        ValueError: If any of the following occurs:
            - Matrix doesn't have a dominant diagonal (no convergence guarantee)
            - Any diagonal element is zero (would cause division by zero)
            - Maximum number of iterations is exceeded without convergence

    Note:
        Requires two helper functions:
        1. dominant_diagonal(): Checks if matrix is diagonally dominant
        2. absolute_distance() and relative_distance(): Calculate convergence criteria
    """
    if not dominant_diagonal(A):
        raise ValueError("Does not guarantee convergence")
    
    if np.any(np.diag(A) == 0):
        raise ValueError("Zero on diagonal - cannot divide by zero")
    
    n = A.shape[0]
    x = np.array([b[i]/A[i, i] for i in range(n)])
    
    for _ in range(max_iterations):
        x_old = x.copy()
        
        for i in range(n):
            sum = np.sum(A[i, :i] @ x_old[:i]) + (A[i, i+1:] @ x_old[i+1:])
            x[i] = (b[i] - sum) / A[i, i]
        
        if absolute_distance(x,x_old) < epsilon or relative_distance(x,x_old,epsilon) < epsilon:
            return x
    
    raise ValueError("Exceeded the maximum number of iterations")

### Gauss-Seidel

In [125]:
def gauss_seidel_method(A: np.ndarray, b: np.ndarray, max_iterations: int = 100, epsilon: float = sys.float_info.epsilon) -> np.ndarray:
    """Solves a system of linear equations using the Gauss-Seidel iterative method.

    The Gauss-Seidel method is an iterative technique for solving a square system of n linear
    equations with unknown x. The method will converge if the matrix is either strictly diagonally
    dominant or symmetric and positive definite.

    Args:
        A: A square numpy array representing the coefficient matrix of the linear system.
            Must be strictly diagonally dominant for convergence guarantee.
        b: A numpy array representing the right-hand side vector of the linear system.
        max_iterations: Maximum number of iterations to perform before giving up.
            Defaults to 100.
        epsilon: The tolerance for determining convergence. The algorithm stops when either
            the absolute or relative distance between iterations is less than epsilon.
            Defaults to system's float epsilon.

    Returns:
        A numpy array representing the solution vector x that satisfies Ax = b.

    Raises:
        ValueError: If the matrix doesn't have a dominant diagonal (no convergence guarantee),
            if any diagonal element is zero (would cause division by zero),
            or if the maximum number of iterations is exceeded without convergence.

    Note:
        The function uses two helper functions:
        - dominant_diagonal(): Checks if matrix is diagonally dominant
        - absolute_distance() and relative_distance(): Calculate convergence criteria
    """
    if not dominant_diagonal(A):
        raise ValueError("Does not guarantee convergence")
    
    if np.any(np.diag(A) == 0):
        raise ValueError("Zero on diagonal - cannot divide by zero")
    
    n = A.shape[0]
    x = np.array([b[i]/A[i, i] for i in range(n)])
    
    for _ in range(max_iterations):
        x_old = x.copy()
        
        for i in range(n):
            sum = np.sum(A[i, :i] @ x[:i]) + (A[i, i+1:] @ x[i+1:])
            x[i] = (b[i] - sum) / A[i, i]
        
        if absolute_distance(x,x_old) < epsilon or relative_distance(x,x_old,epsilon) < epsilon:
            return x
    
    raise ValueError("Exceeded the maximum number of iterations")

### Test

In [126]:
# Matrix A
print(f"Matrix A:\n{A}\nb-vector:\n{B3}\n")

print("Gauss-Elimination Method:")
x = gauss_elimination_method(A, B3)
print(f"Solution: {x}")
print(f"Residue: {residue(A,B3,x)}\n")

print("LU Decomposition Method:")
x = lu_method(A, B3)
print(f"Solution: {x}")
print(f"Residue: {residue(A, B3, x)}\n")

print("Gauss-Jacobi Method:")
x = gauss_jacobi_method(A, B3, epsilon=epsilon)
print(f"Solution: {x}")
print(f"Residue: {residue(A,B3,x)}\n")

print("Gauss-Seidel Method:")
x = gauss_seidel_method(A, B3, epsilon=epsilon)
print(f"Solution: {x}")
print(f"Residue: {residue(A,B3,x)}\n")

Matrix A:
[[10  2  1]
 [ 1  5  1]
 [ 2  3 10]]
b-vector:
[ 7 -8  6]

Gauss-Elimination Method:
Solution: [ 1. -2.  1.]
Residue: 0.0

LU Decomposition Method:
Solution: [ 1. -2.  1.]
Residue: 0.0

Gauss-Jacobi Method:
Solution: [ 1. -2.  1.]
Residue: 0.0

Gauss-Seidel Method:
Solution: [ 1. -2.  1.]
Residue: 0.0



In [127]:
# Matrix F
print(f"Matrix F:\n{F}\nb-vector:\n{B4}\n")

print("Gauss-Elimination Method:")
x = gauss_elimination_method(F, B4)
print(f"Solution: {x}")
print(f"Residue: {residue(F,B4,x)}\n")

print("LU Decomposition Method:")
x = lu_method(F, B4)
print(f"Solution: {x}")
print(f"Residue: {residue(F,B4,x)}\n")

print("Gauss-Jacobi Method:")
x = gauss_jacobi_method(F, B4, epsilon=epsilon)
print(f"Solution: {x}")
print(f"Residue: {residue(F,B4,x)}\n")

print("Gauss-Seidel Method:")
x = gauss_seidel_method(F, B4, epsilon=epsilon)
print(f"Solution: {x}")
print(f"Residue: {residue(F,B4,x)}\n")

Matrix F:
[[32  5  7  0]
 [ 3 30  6  1]
 [ 7  5 54  8]
 [ 1  9  8 30]]
b-vector:
[1 2 3 4]

Gauss-Elimination Method:
Solution: [0.01554328 0.0550245  0.03249894 0.10764149]
Residue: 0.0

LU Decomposition Method:
Solution: [0.01554328 0.0550245  0.03249894 0.10764149]
Residue: 0.0

Gauss-Jacobi Method:
Solution: [0.01554328 0.0550245  0.03249894 0.10764149]
Residue: 8.881784197001252e-16

Gauss-Seidel Method:
Solution: [0.01554328 0.0550245  0.03249894 0.10764149]
Residue: 2.220446049250313e-16



In [128]:
# Matrix C
print(f"Matrix C:\n{C}\nb-vector:\n{B5}\n")

print("Gauss-Elimination Method:")
x = gauss_elimination_method(C, B5)
print(f"Solution: {x}")
print(f"Residue: {residue(C,B5,x)}\n")

print("LU Decomposition Method:")
x = lu_method(C, B5)
print(f"Solution: {x}")
print(f"Residue: {residue(C,B5,x)}\n")

print("Gauss-Jacobi Method:")
x = gauss_jacobi_method(C, B5, epsilon=epsilon)
print(f"Solution: {x}")
print(f"Residue: {residue(C,B5,x)}\n")

print("Gauss-Seidel Method:")
x = gauss_seidel_method(C, B5, epsilon=epsilon)
print(f"Solution: {x}")
print(f"Residue: {residue(C,B5,x)}\n")

Matrix C:
[[45  2  5  6  0]
 [ 9 37  1  7  3]
 [ 6  1 49  8  5]
 [ 8  9  7 55  1]
 [ 0  0  3  2 31]]
b-vector:
[1 2 3 4 5]

Gauss-Elimination Method:
Solution: [0.00926115 0.02706719 0.03404403 0.05981566 0.15413666]
Residue: 8.881784197001252e-16

LU Decomposition Method:
Solution: [0.00926115 0.02706719 0.03404403 0.05981566 0.15413666]
Residue: 8.881784197001252e-16

Gauss-Jacobi Method:
Solution: [0.00926115 0.02706719 0.03404403 0.05981566 0.15413666]
Residue: 1.7763568394002505e-15

Gauss-Seidel Method:
Solution: [0.00926115 0.02706719 0.03404403 0.05981566 0.15413666]
Residue: 8.881784197001252e-16



## Interpolations:

Techniques for estimating unknown values between points

### Overall parameters

In [129]:
x_points = np.array([-1, 0, 2])
y_points = np.array([4, 1, -1])

### Linear System Method

#### Vandemond Matrix Function

In [ ]:
def vandermonde_matrix(x):
        degree = len(x) - 1
        return np.array([[xi**j for j in range(degree + 1)] for xi in x])

#### Linear Method

In [ ]:
def linear_method(x_points: np.ndarray, y_points: np.ndarray):
    if x_points.shape[0] != y_points.shape[0]:
            raise ValueError("x_points and y_points have different lengths")
    V_matrix = vandermonde_matrix(x_points)
    A_points = lu_method(V_matrix, y_points)
    return A_points
    

### Lagrange Method

In [ ]:
def lagrange_method(x_points: np.ndarray, y_points: np.ndarray) -> Callable[[float], float]:
    """Constructs and returns the Lagrange interpolation polynomial for given data points.

    The Lagrange interpolation polynomial is the unique polynomial of least degree that
    interpolates a given set of points (x_i, y_i) and passes through all of them.

    Args:
        x_points: A 1-D numpy array of x-coordinates of the data points (interpolation nodes).
            Must not contain duplicate values.
        y_points: A 1-D numpy array of y-coordinates of the data points corresponding to x_points.
            Must have the same length as x_points.

    Returns:
        A callable function that represents the Lagrange interpolation polynomial.
        The function takes a float value x and returns the interpolated value at x.

    Raises:
        ValueError: If x_points and y_points have different lengths
    """
    if x_points.shape[0] != y_points.shape[0]:
            raise ValueError("x_points and y_points have different lengths")
    def polynomial(x: float) -> float:
        """Evaluates the Lagrange interpolation polynomial at point x.

        Args:
            x: The point at which to evaluate the interpolation polynomial.

        Returns:
            The interpolated value at point x.

        Note:
            This is the actual implementation of the Lagrange interpolation formula:
            L(x) = Σ [y_i * ℓ_i(x)] where ℓ_i(x) is the i-th Lagrange basis polynomial.
        """
        
        n = x_points.shape[0]
        result = 0.0
        for i in range(n):
            prod = 1.0
            for j in range(n):
                if i == j:
                    continue
                prod *= (x - x_points[j]) / (x_points[i] - x_points[j])
            result += y_points[i] * prod
        return result
    return polynomial

TypeError: 'builtin_function_or_method' object is not subscriptable

### Tests

In [ ]:
polynomial = lagrange_method(x_points=x_points, y_points=y_points)
print(polynomial(-1))
print(polynomial(0))
print(polynomial(2))
print(polynomial(1))

4.0
1.0
-1.0
-0.6666666666666665
